<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/2_basic_project_examples/5_extracting_information_from_unstructured_data/5_extracting_information_from_unstructured_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Messy Text In, Structured JSON Out

Support tickets, emails, and reports arrive as free-form text. You give the model a **schema**; it hands back **typed, validated JSON** that matches it exactly, no regex or manual parsing required.

## Setup

Install the libraries and connect the client to your API key.

In [1]:
%pip install -q openai==2.28.0 pydantic==2.12.3

import os
from openai import OpenAI

# Key from Colab secrets, an environment variable, or a prompt, in that order.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
TEXT_MODEL = "gpt-5-nano"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/robertbarcik/git-repos/ADK-tutorial/.venv/bin/python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## The schema

A Pydantic model describing exactly what you want back. This is the only thing that changes when you extract something else.

In [2]:
from enum import Enum
from pydantic import BaseModel


class Urgency(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class SupportTicket(BaseModel):
    user_name: str
    department: str
    issue_summary: str
    urgency: Urgency
    category: str

## The one thing you change

Some messy, real-world text. Swap it for anything and the extraction below still works.

In [3]:
email = """
From: jennifer.martinez@company.com
Subject: Urgent - Cannot Access Email

Hi IT Support,

I'm Jennifer Martinez from the Sales department. Since this morning I can't
log into Outlook, it just keeps saying my password is wrong even though I
haven't changed it. I have a client call in an hour and I need my inbox.
Please help ASAP!

Thanks,
Jennifer
"""

## The one call

This is the whole engine. `text_format` tells the model the exact shape to return; `output_parsed` gives you back a real, validated `SupportTicket` object.

In [4]:
ticket = client.responses.parse(
    model=TEXT_MODEL,
    input=f"Extract the support ticket details from this email:\n\n{email}",
    text_format=SupportTicket,
).output_parsed

print(ticket)

user_name='Jennifer Martinez' department='Sales' issue_summary='Cannot log into Outlook; password shows as incorrect despite not being changed.' urgency=<Urgency.HIGH: 'high'> category='Authentication / Login'


## Now try your own text

Paste anything messy below (an email, a Slack message, a report) and run the same extraction on it.

In [5]:
your_text = """
Hey, this is Marcus from Engineering. My laptop screen keeps flickering
and sometimes goes black for a few seconds, it's been happening for two
days and it's getting worse. Not urgent but annoying, can someone take
a look this week?
"""

your_ticket = client.responses.parse(
    model=TEXT_MODEL,
    input=f"Extract the support ticket details from this email:\n\n{your_text}",
    text_format=SupportTicket,
).output_parsed

print(your_ticket)

user_name='Marcus' department='Engineering' issue_summary='Laptop screen flickers and occasionally goes black for a few seconds; started two days ago and is getting worse.' urgency=<Urgency.LOW: 'low'> category='Display (Laptop screen)'


## Take it further

- **Batch it.** Loop the same call over a list of emails and build a table of tickets.
- **Route it.** Use `ticket.urgency` and `ticket.category` to send each ticket to the right queue.
- **Grade its confidence.** Add a `confidence: float` field to the schema and flag anything below a threshold for human review.

The schema is the only thing that changes; the call stays the same.

### Your turn

Edit `SupportTicket` (add or rename a field) or swap in your own messy text, and re-run.